# Tema de programare: Validarea automata a datelor


Bun venit la tema despre **Validarea automata a datelor**!

In pipeline-urile de date din productie, problemele tacite de calitate a datelor sunt printre cele mai greu de depistat buguri - o coloana care nu ar trebui sa fie niciodata nula are brusc 5 % valori lipsa, un camp numeric depaseste discret intervalul valid sau ID-uri duplicate trec neobservate. **Verificarile automate de validare** rezolva aceasta problema prin codificarea asteptarilor tale despre date sub forma de asertiuni executabile, care ruleaza de fiecare data cand datele trec prin pipeline-ul tau.

Vom construi aceste verificari **de la zero folosind doar pandas** - nu sunt necesare framework-uri externe.

### De ce conteaza validarea automata

| Problema | Efect tacut | Verificare automata |
|---------|---------------|-----------------|
| Valori nule intr-o cheie de join | Randuri lipsa dupa merge | **Verificare de null** |
| Varsta = -5 sau 999 | Statistici distorsionate, model gresit | **Verificare de interval** |
| ID-uri duplicate | Inregistrari numarate de doua ori | **Verificare de unicitate** |
| Email-uri malformate | Trimiteri esuate, pierdere de venit | **Verificare de pattern/regex** |
| Incalcari multiple ale regulilor | Erori in lant in etapele ulterioare | **Motor de reguli** |

### Ce vei implementa

* **`validate_no_nulls`** - detecteaza valorile nule coloana cu coloana
* **`validate_range`** - marcheaza valorile numerice din afara unui interval permis
* **`validate_uniqueness`** - numara valorile duplicate dintr-o coloana
* **`validate_regex`** - verifica valorile de tip sir fata de un pattern regex
* **`create_validation_report`** - orchestreaza toate verificarile dintr-o lista declarativa de reguli

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt inghetate, cu exceptia celor in care trebuie sa trimiti solutiile tale sau atunci cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde trebuie sa scrii codul solutiei. **Nu adauga si nu modifica niciun cod care este in afara acestor comentarii**.

* Poti adauga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, asa ca nu te baza pe celulele nou create pentru a gazdui codul solutiei tale; foloseste locurile oferite pentru asta.

---


## Cuprins
- [Importuri](#imports)
- [1 - Intelegerea datelor](#1---understanding-the-data)
- [2 - Validarea valorilor nule](#2---null-validation)
    - **[Exercitiul 1 - `validate_no_nulls`](#exercise-1---validate_no_nulls)**
- [3 - Validarea intervalului](#3---range-validation)
    - **[Exercitiul 2 - `validate_range`](#exercise-2---validate_range)**
- [4 - Validarea unicitatii](#4---uniqueness-validation)
    - **[Exercitiul 3 - `validate_uniqueness`](#exercise-3---validate_uniqueness)**
- [5 - Validarea pattern-ului](#5---pattern-validation)
    - **[Exercitiul 4 - `validate_regex`](#exercise-4---validate_regex)**
- [6 - Motor de reguli de validare](#6---validation-rule-engine)
    - **[Exercitiul 5 - `create_validation_report`](#exercise-5---create_validation_report)**
- [7 - Vizualizarea rezultatelor](#7---visualizing-results)


<a name='imports'></a>
## Importuri


In [ ]:
import re
import numpy as np
import pandas as pd

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-the-data'></a>
## 1 - Intelegerea datelor

Vom lucra cu un **set de date sintetic de tip survey** care a fost corupt in mod deliberat cu o rata controlata de erori. Setul de date contine urmatoarele coloane:

| Coloana | Tip | Interval / format valid | Erori injectate |
|--------|------|---------------------|-----------------|
| `respondent_id` | string | unic, format `R-XXXX` | ID-uri duplicate |
| `age` | int | 0 - 120 | valori nule, valori in afara intervalului |
| `score` | float | 0 - 100 | valori nule, valori in afara intervalului |
| `email` | string | pattern `*@*.*` | adrese malformate |
| `country` | string | orice | valori nule |

Cu `error_rate=0.1`, aproximativ 10 % dintre randurile fiecarei coloane vor contine o problema injectata. Functiile tale de validare trebuie sa detecteze aceste probleme in mod fiabil.


In [ ]:
df = helper_utils.generate_survey_dataset(n_samples=300, error_rate=0.1, random_state=42)
print(f"Shape: {df.shape}")
print(f"\nColumn info:")
df.info()
print(f"\nBasic statistics:")
df.describe().round(2)

<a name='2---null-validation'></a>
## 2 - Validarea valorilor nule

<a name='exercise-1---validate_no_nulls'></a>
### **Exercitiul 1 - `validate_no_nulls`**

**Sarcina ta:**
Implementeaza o functie care inspecteaza fiecare coloana ceruta si raporteaza cate valori nule contine.

**Cerinte:**
- Accepta un `pd.DataFrame` si o lista optionala `columns`.
- Cand `columns` este `None`, verifica **toate** coloanele.
- Returneaza un dict care mapeaza fiecare nume de coloana la un sub-dict cu cheile `'passed'` (`bool`) si `'null_count'` (`int`).
- `'passed'` este `True` doar atunci cand `null_count == 0`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Daca `columns` este `None`, seteaz-o la `df.columns.tolist()`.
2. Creeaza un dict gol `result`.
3. Parcurge fiecare nume de coloana din `columns`.
4. Calculeaza `null_count = int(df[col].isna().sum())`.
5. Stocheaza `result[col] = {'passed': null_count == 0, 'null_count': null_count}`.
6. Returneaza `result`.

</details>


In [ ]:
# GRADED FUNCTION: validate_no_nulls
def validate_no_nulls(df, columns=None):
    """
    Check each column for null values.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list): Columns to check (default: all).

    Returns:
        dict: {column_name: {'passed': bool, 'null_count': int}}
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
null_report = validate_no_nulls(df)
print("Null Validation Report:")
for col, info in null_report.items():
    status = "\u2713 PASSED" if info['passed'] else "\u2717 FAILED"
    print(f"  {col}: {status} \u2014 {info['null_count']} nulls")

In [ ]:
unittests.exercise_1(validate_no_nulls)

<a name='3---range-validation'></a>
## 3 - Validarea intervalului

<a name='exercise-2---validate_range'></a>
### **Exercitiul 2 - `validate_range`**

**Sarcina ta:**
Implementeaza o functie care verifica daca toate valorile numerice non-nule dintr-o coloana se afla in interiorul unui interval specificat `[min_val, max_val]`.

**Cerinte:**
- Elimina valorile nule inainte de verificare (gestionarea null-urilor este treaba Exercitiului 1).
- `min_val=None` inseamna fara limita inferioara; `max_val=None` inseamna fara limita superioara.
- Returneaza un dict cu cheile `'passed'` (`bool`), `'violations'` (`int`) si `'violation_pct'` (`float`, rotunjit la 2 zecimale).

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Izoleaza valorile non-nule: `col_data = df[column].dropna()`.
2. Incepe cu o masca booleana formata doar din `True`: `mask = pd.Series(True, index=col_data.index)`.
3. Daca `min_val is not None`, combina masca cu `col_data >= min_val` folosind `AND`.
4. Daca `max_val is not None`, combina masca cu `col_data <= max_val` folosind `AND`.
5. Numara incalcarile: `violations = int((~mask).sum())`.
6. Calculeaza `violation_pct = round(violations / len(col_data) * 100, 2)` (protejeaza-te impotriva impartirii la zero).

</details>


In [ ]:
# GRADED FUNCTION: validate_range
def validate_range(df, column, min_val=None, max_val=None):
    """
    Validate that numeric values fall within a specified range.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to validate.
        min_val (float): Minimum allowed value (None = no lower bound).
        max_val (float): Maximum allowed value (None = no upper bound).

    Returns:
        dict: {'passed': bool, 'violations': int, 'violation_pct': float}
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
age_report = validate_range(df, 'age', min_val=0, max_val=120)
score_report = validate_range(df, 'score', min_val=0, max_val=100)
print(f"Age range validation:   {age_report}")
print(f"Score range validation: {score_report}")

In [ ]:
unittests.exercise_2(validate_range)

<a name='4---uniqueness-validation'></a>
## 4 - Validarea unicitatii

<a name='exercise-3---validate_uniqueness'></a>
### **Exercitiul 3 - `validate_uniqueness`**

**Sarcina ta:**
Implementeaza o functie care numara cate valori dintr-o coloana apar de mai multe ori (adica sunt duplicate).

**Cerinte:**
- Foloseste `pd.Series.duplicated()` - aceasta marcheaza fiecare aparitie de dupa prima ca duplicat.
- Returneaza un dict cu cheile `'passed'` (`bool`) si `'duplicate_count'` (`int`).
- `'passed'` este `True` doar atunci cand `duplicate_count == 0`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Apeleaza `df[column].duplicated()` - aceasta returneaza un Series boolean unde `True` inseamna ca valoarea este un duplicat.
2. Sumeaza Series-ul boolean pentru a numara duplicatele: `duplicate_count = int(df[column].duplicated().sum())`.
3. Returneaza `{'passed': duplicate_count == 0, 'duplicate_count': duplicate_count}`.

</details>


In [ ]:
# GRADED FUNCTION: validate_uniqueness
def validate_uniqueness(df, column):
    """
    Check for duplicate values in a column.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to check for uniqueness.

    Returns:
        dict: {'passed': bool, 'duplicate_count': int}
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
id_report = validate_uniqueness(df, 'respondent_id')
print(f"ID uniqueness: {id_report}")

In [ ]:
unittests.exercise_3(validate_uniqueness)

<a name='5---pattern-validation'></a>
## 5 - Validarea pattern-ului

<a name='exercise-4---validate_regex'></a>
### **Exercitiul 4 - `validate_regex`**

**Sarcina ta:**
Implementeaza o functie care valideaza valorile de tip sir dintr-o coloana fata de un pattern de expresie regulata.

**Cerinte:**
- Elimina valorile nule inainte de verificare.
- Converteste valorile ramase la `str` inainte de potrivire.
- Foloseste `pd.Series.str.match(pattern)` - aceasta verifica o potrivire **la inceputul** fiecarui sir.
- Returneaza un dict cu cheile `'passed'` (`bool`) si `'violations'` (`int`).

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Izoleaza valorile de tip sir non-nule: `col_data = df[column].dropna().astype(str)`.
2. Aplica pattern-ul: `matches = col_data.str.match(pattern)`.
3. Numara randurile care nu se potrivesc: `violations = int((~matches).sum())`.
4. Returneaza `{'passed': violations == 0, 'violations': violations}`.

**Sfat:** `str.match` ancoreaza la inceput. Pentru a valida intregul sir (de ex. formatul de email `.+@.+\..+`), asigura-te ca pattern-ul tau acopera intreaga valoare sau foloseste `str.fullmatch` daca este necesar.

</details>


In [ ]:
# GRADED FUNCTION: validate_regex
def validate_regex(df, column, pattern):
    """
    Validate that string values in a column match a regex pattern.

    Args:
        df (pd.DataFrame): Input DataFrame.
        column (str): Column to validate.
        pattern (str): Regex pattern each value must match.

    Returns:
        dict: {'passed': bool, 'violations': int}
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
email_report = validate_regex(df, 'email', pattern=r'.+@.+\..+')
print(f"Email format validation: {email_report}")

In [ ]:
unittests.exercise_4(validate_regex)

<a name='6---validation-rule-engine'></a>
## 6 - Motor de reguli de validare

<a name='exercise-5---create_validation_report'></a>
### **Exercitiul 5 - `create_validation_report`**

**Sarcina ta:**
Implementeaza un **motor de reguli** care accepta o lista de configuratii pentru reguli de validare si ruleaza toate cele patru tipuri de verificari construite mai sus, returnand un raport structurat unificat.

**Cerinte:**
- Fiecare dict de regula trebuie sa aiba cel putin o cheie `'type'` si o cheie `'column'`.
- Tipuri de reguli suportate: `'no_nulls'`, `'range'`, `'uniqueness'`, `'regex'`.
- Pentru regulile `'range'`, citeste si cheile optionale `'min_val'` si `'max_val'`.
- Pentru regulile `'regex'`, citeste si cheia `'pattern'`.
- Returneaza o `list` de dict-uri rezultat, cate unul per regula, fiecare continand: `'rule'`, `'column'`, `'passed'`, `'violations'`.

<details>
  <summary><b><font color="green">Indicii suplimentare de cod (Click pentru extindere)</font></b></summary>

**Ghid pas cu pas:**

1. Initializeaza o lista goala `report = []`.
2. Parcurge fiecare dict `rule` din `rules`.
3. Extrage `rule_type = rule['type']` si `column = rule['column']`.
4. Creeaza un dict de baza `entry`: `{'rule': rule_type, 'column': column, 'passed': True, 'violations': 0}`.
5. Trimite catre validatorul corect pe baza lui `rule_type`:
   - `'no_nulls'` → apeleaza `validate_no_nulls(df, [column])`, citeste `res[column]`.
   - `'range'` → apeleaza `validate_range(df, column, min_val=rule.get('min_val'), max_val=rule.get('max_val'))`.
   - `'uniqueness'` → apeleaza `validate_uniqueness(df, column)`, citeste `res['duplicate_count']`.
   - `'regex'` → apeleaza `validate_regex(df, column, pattern=rule['pattern'])`.
6. Actualizeaza `entry['passed']` si `entry['violations']` din rezultat.
7. Adauga `entry` in `report`.
8. Returneaza `report`.

</details>


In [ ]:
# GRADED FUNCTION: create_validation_report
def create_validation_report(df, rules):
    """
    Run a list of validation rules and return a structured report.

    Args:
        df (pd.DataFrame): Input DataFrame.
        rules (list): List of rule dicts. Supported types:
            - {'type': 'no_nulls', 'column': str}
            - {'type': 'range', 'column': str, 'min_val': float, 'max_val': float}
            - {'type': 'uniqueness', 'column': str}
            - {'type': 'regex', 'column': str, 'pattern': str}

    Returns:
        list: List of result dicts with keys 'rule', 'column', 'passed', 'violations'.
    """
    ### ÎNCEPUT DE COD AICI ###

    
    raise NotImplementedError("Implement this function")

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
rules = [
    {'type': 'no_nulls', 'column': 'age'},
    {'type': 'range', 'column': 'age', 'min_val': 0, 'max_val': 120},
    {'type': 'range', 'column': 'score', 'min_val': 0, 'max_val': 100},
    {'type': 'uniqueness', 'column': 'respondent_id'},
    {'type': 'regex', 'column': 'email', 'pattern': r'.+@.+\..+'},
]

report = create_validation_report(df, rules)
report_df = pd.DataFrame(report)
print("Validation Report:")
print(report_df.to_string(index=False))

In [ ]:
unittests.exercise_5(create_validation_report)

<a name='7---visualizing-results'></a>
## 7 - Vizualizarea rezultatelor


In [ ]:
# Build validation_report dict format for visualization
validation_dict = {row['rule'] + '_' + row['column']: {'passed': row['passed'], 'violations': row['violations']}
                   for row in report}
helper_utils.visualize_validation_results(validation_dict)